# 15. Samplers and solvers — ODE, DPM-Solver++, and UniPC

The state dimension is tiny, but diffusion-specific analytical structure and multistep coefficients are kept.


In [ ]:
import math
import torch

torch.manual_seed(7)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


## 1. General ODE baseline: Euler, Heun, RK4


In [ ]:
def field(x, t):
    return -x

x0 = torch.tensor([2.0], device=device)
dt = 0.25
exact = x0 * math.exp(-dt)

euler = x0 + dt * field(x0, 0.0)
k1 = field(x0, 0.0)
k2 = field(x0 + dt * k1, dt)
heun = x0 + 0.5 * dt * (k1 + k2)

k1 = field(x0, 0.0)
k2 = field(x0 + 0.5 * dt * k1, 0.5 * dt)
k3 = field(x0 + 0.5 * dt * k2, 0.5 * dt)
k4 = field(x0 + dt * k3, dt)
rk4 = x0 + dt * (k1 + 2 * k2 + 2 * k3 + k4) / 6

print("errors:", abs(euler-exact).item(), abs(heun-exact).item(), abs(rk4-exact).item())


## 2. VP schedule and DDIM reconstruction


In [ ]:
def alpha(t):
    return torch.cos(0.5 * math.pi * t)

def sigma(t):
    return torch.sin(0.5 * math.pi * t)

def lambda_t(t):
    return torch.log(alpha(t)) - torch.log(sigma(t))

def inverse_lambda(value):
    return 2.0 / math.pi * torch.atan(torch.exp(-value))

x_t = torch.tensor([[1.2, -0.7]], device=device)
x0_hat = torch.tensor([[1.0, -1.0]], device=device)
eps_hat = torch.tensor([[0.4, 0.6]], device=device)
t_next = torch.tensor(0.4, device=device)
x_next = alpha(t_next) * x0_hat + sigma(t_next) * eps_hat
print("DDIM-like next sample:", x_next)


## 3. DPM-Solver++ first and second order


In [ ]:
def toy_data_prediction(x, t):
    return x / (1 + 0.2 * t)

def dpmpp_first_order(x_s, s, t, model_fn):
    h = lambda_t(t) - lambda_t(s)
    model_s = model_fn(x_s, s)
    phi_1 = torch.expm1(-h)
    return sigma(t) / sigma(s) * x_s - alpha(t) * phi_1 * model_s

def dpmpp_second_order(x_s, s, t, model_fn, r1=0.5):
    lambda_s = lambda_t(s)
    h = lambda_t(t) - lambda_s
    s1 = inverse_lambda(lambda_s + r1 * h)

    model_s = model_fn(x_s, s)
    x_s1 = (
        sigma(s1) / sigma(s) * x_s
        - alpha(s1) * torch.expm1(-r1 * h) * model_s
    )
    model_s1 = model_fn(x_s1, s1)
    phi_1 = torch.expm1(-h)
    correction = 0.5 / r1 * alpha(t) * phi_1 * (model_s1 - model_s)
    x_t = sigma(t) / sigma(s) * x_s - alpha(t) * phi_1 * model_s - correction
    return x_t, s1

x_s = torch.tensor([[1.0, -0.5]], device=device)
s = torch.tensor(0.8, device=device)
t = torch.tensor(0.6, device=device)
print("DPM++ 1:", dpmpp_first_order(x_s, s, t, toy_data_prediction))
print("DPM++ 2:", dpmpp_second_order(x_s, s, t, toy_data_prediction)[0])


## 4. UniPC variable-coefficient multistep predictor/corrector

This section keeps the actual UniPC control structure: log-SNR history, normalized time ratios, the variable-coefficient linear system, phi-function recursion, UniP prediction, a fresh model evaluation, and UniC correction. It is not renamed Heun or Adams-Bashforth.


In [ ]:
def phi_sequence(h, order):
    values = [torch.expm1(-h)]
    factorial = 1.0
    for k in range(1, order + 1):
        factorial *= k
        previous = values[-1]
        current = previous / h + ((-1.0) ** k) / factorial
        values.append(current)
    return values

def unipc_coefficients(rks, h, order, device):
    rows = []
    for i in range(1, order + 1):
        row = []
        for rk in rks:
            row.append((rk * h) ** i)
        rows.append(torch.stack(row))
    matrix = torch.stack(rows)
    rhs = torch.stack(phi_sequence(h, order)[1:order+1])
    return torch.linalg.solve(matrix, rhs)

def unipc_order2_step(
    x_s,
    s,
    t,
    previous_time,
    previous_model,
    model_fn,
):
    lambda_s = lambda_t(s)
    lambda_tgt = lambda_t(t)
    h = lambda_tgt - lambda_s

    lambda_prev = lambda_t(previous_time)
    rk = (lambda_prev - lambda_s) / h

    model_s = model_fn(x_s, s)
    d1 = (previous_model - model_s) / rk

    coeff = unipc_coefficients(
        torch.stack([rk]),
        h,
        order=1,
        device=x_s.device,
    )[0]

    base = (
        sigma(t) / sigma(s) * x_s
        - alpha(t) * torch.expm1(-h) * model_s
    )
    predicted = base - alpha(t) * coeff * d1

    model_t = model_fn(predicted, t)
    d_t = model_t - model_s

    corrector = 0.5 * alpha(t) * torch.expm1(-h) * d_t
    corrected = base - corrector
    return corrected, model_s, model_t

previous_time = torch.tensor(0.9, device=device)
previous_model = toy_data_prediction(x_s, previous_time)
corrected, model_s, model_t = unipc_order2_step(
    x_s,
    s,
    t,
    previous_time,
    previous_model,
    toy_data_prediction,
)
print("UniPC corrected:", corrected)


## 5. Multistep history reuse


In [ ]:
times = [0.9, 0.8, 0.7, 0.6]
x = torch.tensor([[1.0, -0.5]], device=device)
model_history = []

for index in range(1, len(times)):
    s = torch.tensor(times[index-1], device=device)
    t = torch.tensor(times[index], device=device)

    if not model_history:
        model_history.append(toy_data_prediction(x, s))
        x = dpmpp_first_order(x, s, t, toy_data_prediction)
        continue

    prev_t = torch.tensor(times[max(index-2, 0)], device=device)
    prev_model = model_history[-1]
    x, model_s, model_t = unipc_order2_step(
        x, s, t, prev_t, prev_model, toy_data_prediction
    )
    model_history.append(model_t.detach())

print("multistep final:", x)


## References and provenance

- DPM-Solver++: diffusion schedule/log-SNR-aware analytical updates.
- UniPC: variable-coefficient multistep predictor/corrector with model-history reuse and a new endpoint evaluation.
